# Hard two-stage baseline — four-dataset, dataset-blind inference

This notebook trains the non-MoE baseline used for the paper comparison. A standalone dataset-identity classifier first predicts which source dataset a sample belongs to (phase A). Its argmax prediction hard-routes that sample to one independently trained per-dataset encoder+head (phase B). Inference never receives the ground-truth dataset ID. There is no shared representation, joint optimization, or probability blending between the two phases.

The encoder widths, latent width, activation, dropout, optimizer settings, data split, preprocessing, and task vocabulary are set to match the finalized four-dataset experiment. This is intentionally called **hard two-stage**, not two-stage MoE.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'

EXECUTION_MODE = 'out_of_core_full'  # out_of_core_full | in_memory_smoke
ACTIVE_DATASETS = [
    'NF-UNSW-NB15-v3',
    'NF-ToN-IoT-v3',
    'NF-BoT-IoT-v3',
    'NF-CICIDS2018-v3',
]
RUN_NAME = 'nfv3_4way_hard_two_stage_comparable_seed0_v1'
REFERENCE_MOE_RUN_NAME = 'nfv3_4way_moe_basic_seed0_v1'  # notebook 12 run
SEED = 0
LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
DROPOUT = 0.2
EPOCHS_PHASE_A = 30
EPOCHS_PHASE_B = 30
BATCH_SIZE = 512
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0
FORCE_RESTART = False
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment

The token is sent as a temporary HTTP header and is not stored in the clone URL or Git configuration.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
}
repo_dir = Path('/content') / GITHUB_REPO
repo_url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo_dir)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository:', repo_dir)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## Preflight and fairness contract

Phase A and every phase-B classifier receive separate parameter objects. Phase B uses the same encoder MLP shape for every dataset; no gradient can cross from one dataset classifier to another.

In [ ]:
import torch
from data.registry import get_spec
if EXECUTION_MODE not in {'out_of_core_full', 'in_memory_smoke'}:
    raise ValueError('Unknown EXECUTION_MODE')
if len(ACTIVE_DATASETS) != 4 or len(set(ACTIVE_DATASETS)) != 4:
    raise ValueError('The paper run requires exactly four distinct source datasets.')
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU and reconnect.')
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError('Missing datasets:\n' + '\n'.join(f'  {name}: {paths}' for name, paths in missing.items()))
if EXECUTION_MODE == 'out_of_core_full':
    aliases = [get_spec(name).feature_alias for name in ACTIVE_DATASETS]
    if any(get_spec(name).kind != 'file' for name in ACTIVE_DATASETS) or any(a != aliases[0] for a in aliases[1:]):
        raise ValueError('Full-data mode requires schema-compatible single-file NF-v3 datasets.')
    if len(aliases[0]) != 47:
        raise ValueError(f'Expected 47 harmonized features, found {len(aliases[0])}.')
if RUN_TESTS:
    subprocess.run([sys.executable, '-u', '-m', 'pytest', '-q'], check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Hard two-stage MLPs:', [47, *ENCODER_HIDDEN_DIMS, LATENT_DIM], '→ task/dataset head')
print('Routing at inference: stage-A argmax prediction only')

## Train and evaluate the two phases

The command uses the existing architecture switch, so returning to the original approach only requires changing `architecture` in another run. Full-data execution is disk-backed and resumable. There is deliberately no phase C.

In [ ]:
import queue, shutil, threading, time
from collections import deque

def run_streaming(command, env, heartbeat_seconds=30):
    process = subprocess.Popen(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    output = queue.Queue()
    def pump_output():
        try:
            for line in process.stdout:
                output.put(line)
        finally:
            output.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    print(f'[notebook] child PID={process.pid}; streaming combined stdout/stderr', flush=True)
    started = time.monotonic()
    try:
        while True:
            try:
                line = output.get(timeout=heartbeat_seconds)
            except queue.Empty:
                status = subprocess.run(
                    ['ps', '-o', 'etime=,%cpu=,%mem=,rss=,stat=', '-p', str(process.pid)],
                    text=True, capture_output=True, check=False,
                ).stdout.strip()
                print(
                    f'[notebook] heartbeat after {(time.monotonic() - started) / 60:.1f} min; '
                    f"PID={process.pid}; etime/cpu%/mem%/rssKB/state={status or 'process exiting'}",
                    flush=True,
                )
                continue
            if line is None:
                break
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        print(f'[notebook] interrupting child PID={process.pid}', flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        raise
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

phase_a_epochs = 1 if EXECUTION_MODE == 'in_memory_smoke' else EPOCHS_PHASE_A
phase_b_epochs = 1 if EXECUTION_MODE == 'in_memory_smoke' else EPOCHS_PHASE_B
effective_force_restart = True if EXECUTION_MODE == 'in_memory_smoke' else FORCE_RESTART
overrides = [
    f'run_name={RUN_NAME}',
    'architecture=hard_two_stage',
    'data.active_datasets=[' + ','.join(ACTIVE_DATASETS) + ']',
    'training.stages=[A,B]',
    'training.device=cuda',
    f'training.force_restart={str(effective_force_restart).lower()}',
    f'training.epochs_a={phase_a_epochs}',
    f'training.epochs_b={phase_b_epochs}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.lr={LEARNING_RATE}',
    f'training.weight_decay={WEIGHT_DECAY}',
    f'model.latent_dim={LATENT_DIM}',
    'model.encoder.hidden_dims=[' + ','.join(map(str, ENCODER_HIDDEN_DIMS)) + ']',
    f'model.encoder.dropout={DROPOUT}',
]
module = 'training.ooc_run' if EXECUTION_MODE == 'out_of_core_full' else 'training.run'
cmd = [sys.executable, '-u', '-m', module, '--config', 'config/default.yaml']
if EXECUTION_MODE == 'in_memory_smoke':
    cmd += ['--mode', 'smoke']
for override in overrides:
    cmd += ['--set', override]
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
run_started = time.monotonic()
checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / RUN_NAME
log_path = checkpoint_dir / 'train.log'
print('Launching unbuffered:', ' '.join(cmd), flush=True)
print('Persistent log:', log_path, flush=True)
try:
    run_streaming(cmd, run_env)
except subprocess.CalledProcessError as exc:
    elapsed = (time.monotonic() - run_started) / 60
    print(f'Training FAILED after {elapsed:.1f} min with exit code {exc.returncode}.', flush=True)
    for label, path in (('Colab local', '/content'), ('Drive output', DRIVE_OUTPUT_DIR)):
        try:
            usage = shutil.disk_usage(path)
            print(f'{label}: {usage.free / 2**30:.1f} GiB free / {usage.total / 2**30:.1f} GiB total')
        except OSError as storage_exc:
            print(f'{label}: storage information unavailable: {storage_exc}')
    if log_path.is_file():
        with log_path.open(errors='replace') as handle:
            print('--- last 80 persistent log lines ---')
            print(''.join(deque(handle, maxlen=80)))
    raise
print(f'Hard two-stage training and evaluation completed in {(time.monotonic() - run_started) / 60:.1f} min.')

## Apples-to-apples result table

The first table is the hard two-stage run. If the reference MoE result exists under `REFERENCE_MOE_RUN_NAME`, the second table joins all headline metrics available in both result schemas. Older result files that predate ROC-AUC reporting are handled without failing. `Gate_By_Dataset.csv` is a routing diagnostic for this baseline: its `stage_a_dataset_accuracy` column measures phase-A identity accuracy and its one-hot route fractions show the argmax commitments.

In [ ]:
import pandas as pd
hard_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / RUN_NAME
moe_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / REFERENCE_MOE_RUN_NAME
hard_overall = pd.read_csv(hard_dir / 'Overall_Metrics.csv')
display(hard_overall)
display(pd.read_csv(hard_dir / 'Gate_By_Dataset.csv'))
reference_path = moe_dir / 'Overall_Metrics.csv'
if reference_path.is_file():
    moe_overall = pd.read_csv(reference_path)
    identity_columns = ['architecture', 'origin', 'rows']
    required_metrics = ['macro_f1']
    optional_metrics = ['roc_auc_ovr_macro', 'balanced_accuracy', 'weighted_f1']
    missing_required = {
        'reference': [c for c in [*identity_columns, *required_metrics] if c not in moe_overall.columns],
        'hard_two_stage': [c for c in [*identity_columns, *required_metrics] if c not in hard_overall.columns],
    }
    missing_required = {name: columns for name, columns in missing_required.items() if columns}
    if missing_required:
        print('Comparison skipped because required columns are missing:', missing_required)
        print('Reference columns:', list(moe_overall.columns))
        print('Hard-two-stage columns:', list(hard_overall.columns))
    else:
        shared_optional = [
            column for column in optional_metrics
            if column in moe_overall.columns and column in hard_overall.columns
        ]
        unavailable = [column for column in optional_metrics if column not in shared_optional]
        if unavailable:
            print('Older/incomplete reference schema; omitted unavailable metrics:', unavailable)
        metrics = [*required_metrics, *shared_optional]
        columns = [*identity_columns, *metrics]
        comparison = pd.concat([
            moe_overall[columns].assign(run=REFERENCE_MOE_RUN_NAME),
            hard_overall[columns].assign(run=RUN_NAME),
        ], ignore_index=True)
        display(comparison.pivot_table(
            index='origin', columns='architecture', values=metrics, aggfunc='last'
        ))
else:
    print('Reference MoE results not found at', moe_dir, '— run notebook 12 or update REFERENCE_MOE_RUN_NAME.')

## Paper-ready method text

**Hard two-stage baseline.** A standalone dataset-identity classifier first predicts which of the four source datasets a sample belongs to (stage a), then hard-routes the sample via argmax to an independently trained per-dataset classifier (stage b). Routing at inference uses the model's own predicted dataset ID rather than a ground-truth label, so the baseline is dataset-blind in the same sense as the MoE. The two stages share no representation or parameters, and routing commits each sample to exactly one stage-b model without blending.